# ASDMotion to 1D-CNN data prep

Windows the annotated pickle into speed+confidence tensors, plus spectral features (V3) and category codes for the 3-way comparison. Run top-to-bottom.

In [1]:
# ── Config ───────────────────────────────────────────────────────────
# Edit these, then run the cells below top-to-bottom.
PKL_PATH     = "dataset.pkl"   # path to the ASDMotion annotated pickle
OUT_DIR      = "prepared"      # arrays + reports are written here
WINDOW       = 90              # frames per sample (3 s @ 30 fps)
STRIDE_TRAIN = 30              # window step for train (overlap = augmentation)
STRIDE_TEST  = 90             # non-overlapping windows for honest evaluation
MAX_GAP      = 4              # interpolate missing runs up to this many frames
MAX_INVALID  = 0.5           # drop a window if > this fraction of it is missing
MAX_SEGMENTS = None          # set to e.g. 500 for a quick dry run, else None

In [2]:
# Imports + constants
import os, json, csv, pickle
from collections import defaultdict
import numpy as np

KP_NAMES = ["nose", "left_eye", "right_eye", "left_ear", "right_ear",
            "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
            "left_wrist", "right_wrist", "left_hip", "right_hip",
            "left_knee", "right_knee", "left_ankle", "right_ankle"]
KEEP = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
KEEP_NAMES = [KP_NAMES[i] for i in KEEP]
L_SH, R_SH, L_HIP, R_HIP = 5, 6, 11, 12

PURE_LOWER = {"Toe walking", "Legs movement"}   # dropped from positives

# Gross-motor (speed-visible) SMM tokens; a positive segment counts as
# gross-motor if its action set intersects this. Everything else positive
# (Fingers, Feeling texture, Playing with object, Tapping, Tremor, Other) is
# "fine-motor" -> dropped from V2's positive class, kept for per-category eval.
GROSS = {"Hand flapping", "Clapping", "Jumping in place", "Spinning in circle",
         "Back and forth", "Body rocking", "Head movement"}


# ----------------------------------------------------------------------
# per-segment scale
# ----------------------------------------------------------------------

In [3]:
# Helper functions
def segment_scale(kp, min_frames=5):
    """Single robust scale (px) for a segment. Returns (scale, method) or (None, 'fail')."""
    def real(i):
        return ~((kp[:, i, 0] == 0) & (kp[:, i, 1] == 0))

    sh = real(L_SH) & real(R_SH)
    if sh.sum() >= min_frames:
        w = np.linalg.norm(kp[sh, L_SH, :] - kp[sh, R_SH, :], axis=1)
        w = w[w > 1e-6]
        if w.size >= min_frames:
            return float(np.median(w)), "shoulder"

    hp = real(L_HIP) & real(R_HIP)
    both = sh & hp
    if both.sum() >= min_frames:
        msh = (kp[:, L_SH, :] + kp[:, R_SH, :]) / 2
        mhip = (kp[:, L_HIP, :] + kp[:, R_HIP, :]) / 2
        tor = np.linalg.norm(msh[both] - mhip[both], axis=1)
        tor = tor[tor > 1e-6]
        if tor.size >= min_frames:
            return float(np.median(tor)), "torso"

    pts = kp[:, KEEP, :]
    realk = ~((pts[..., 0] == 0) & (pts[..., 1] == 0))   # (T,10)
    spreads = []
    for t in range(pts.shape[0]):
        m = realk[t]
        if m.sum() >= 3:
            p = pts[t, m, :]
            spreads.append(max(np.ptp(p[:, 0]), np.ptp(p[:, 1])))
    if spreads:
        s = float(np.median(spreads))
        if s > 1e-6:
            return s, "bbox"
    return None, "fail"


# ----------------------------------------------------------------------
# short-gap linear interpolation
# ----------------------------------------------------------------------
def interp_gaps(coords, valid, max_gap):
    """coords (T,2), valid (T,) bool. Linearly fill interior gaps <= max_gap."""
    T = len(valid)
    c = coords.copy()
    v = valid.copy()
    t = 0
    while t < T:
        if v[t]:
            t += 1
            continue
        s = t
        while t < T and not v[t]:
            t += 1
        e = t                       # invalid run is [s, e)
        gap = e - s
        if s > 0 and e < T and gap <= max_gap:
            x0, x1 = c[s - 1], c[e]
            for j in range(gap):
                a = (j + 1) / (gap + 1)
                c[s + j] = x0 * (1 - a) + x1 * a
                v[s + j] = True
    return c, v


# ----------------------------------------------------------------------
# process one segment -> speed (T,10), conf (T,10), speed_valid (T,10) or None
# ----------------------------------------------------------------------
def process_segment(rec, max_gap):
    kp = np.asarray(rec["keypoint"], dtype=np.float64)      # (T,17,2)
    sc = np.asarray(rec["keypoint_score"], dtype=np.float64)  # (T,17)
    fps = float(rec.get("fps", 30.0))
    T = kp.shape[0]

    scale, method = segment_scale(kp)
    if scale is None:
        return None

    P = kp[:, KEEP, :]                                      # (T,10,2)
    realk = ~((P[..., 0] == 0) & (P[..., 1] == 0))          # (T,10)
    Pn = P / scale

    valid = realk.copy()
    for k in range(len(KEEP)):
        Pn[:, k, :], valid[:, k] = interp_gaps(Pn[:, k, :], realk[:, k], max_gap)

    dt = 1.0 / fps
    speed = np.zeros((T, len(KEEP)), dtype=np.float64)
    both = valid[1:] & valid[:-1]                           # (T-1,10)
    d = np.linalg.norm(Pn[1:] - Pn[:-1], axis=2)            # (T-1,10)
    speed[1:] = np.where(both, d / dt, 0.0)
    speed_valid = np.zeros((T, len(KEEP)), dtype=bool)
    speed_valid[1:] = both

    conf = np.clip(sc[:, KEEP], 0.0, 1.0)
    return speed, conf, speed_valid, method


# ----------------------------------------------------------------------
# window generator
# ----------------------------------------------------------------------
def iter_windows(speed, conf, speed_valid, window, stride, max_invalid):
    """Yield (start, x(C,L), mean_speed, invalid_frac) for windows passing the gate."""
    T = speed.shape[0]
    if T < window:
        return
    for start in range(0, T - window + 1, stride):
        sl = slice(start, start + window)
        sv = speed_valid[sl]                               # (L,10)
        inv = 1.0 - sv.mean()
        if inv > max_invalid:
            continue
        sp = speed[sl]                                     # (L,10)
        cf = conf[sl]                                      # (L,10)
        x = np.concatenate([sp, cf], axis=1).T             # (20, L)
        ms = float(sp[sv].mean()) if sv.any() else 0.0
        yield start, x.astype(np.float32), ms, float(inv)


def subject_of(ident):
    return str(ident).split("_")[0]


def is_pure_lower(action_name, label):
    if label != 1 or not action_name:
        return False
    toks = {t.strip() for t in str(action_name).split(",")}
    return toks.issubset(PURE_LOWER)


# ----------------------------------------------------------------------
# main
# ----------------------------------------------------------------------

In [4]:
def prepare(pkl_path, out_dir, window=90, stride_train=30, stride_test=90,
            max_gap=4, max_invalid=0.5, max_segments=None):
    """Full prep. Writes arrays/reports to out_dir and returns a summary dict.

    New vs the first version:
      * robust speed scaling (clip at train p99.5, standardize by median/IQR)
        so scale-collapse outliers (e.g. 22 bl/s) don't wreck the z-scores;
      * F_<split>.npy : per-window spectral band-energy features (N, 40) for V3;
      * cat_<split>.npy : 0 = NoAction, 1 = gross-motor positive,
        2 = fine-motor positive  -> drives the V2 subset and per-category eval.
    """
    os.makedirs(out_dir, exist_ok=True)
    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)
    anns, split = obj["annotations"], obj["split"]

    split_of = {}
    for name, entries in split.items():
        for e in entries:
            ident = e if isinstance(e, str) else anns[int(e)].get("identifier")
            split_of[ident] = name

    L, n_sp = window, len(KEEP)
    C = 2 * n_sp
    stride = {"train": stride_train, "test": stride_test}

    freqs = np.fft.rfftfreq(L, d=1.0 / 30.0)
    BANDS = [(0.3, 1.0), (1.0, 3.0), (3.0, 6.0), (6.0, 15.0)]
    band_idx = [np.where((freqs > lo) & (freqs <= hi))[0] for lo, hi in BANDS]
    n_spec = n_sp * len(BANDS)

    def spectral(speed_block):
        s = speed_block - speed_block.mean(0, keepdims=True)
        power = np.abs(np.fft.rfft(s, axis=0)) ** 2
        feats = np.stack([power[idx].sum(0) for idx in band_idx], axis=1)
        return np.log1p(feats).ravel().astype(np.float32)

    def category(label, action):
        if label == 0:
            return 0
        toks = {t.strip() for t in str(action).split(",")}
        return 1 if (toks & GROSS) else 2

    counts = defaultdict(int)
    meta = {"train": [], "test": []}
    dropped = defaultdict(int)
    sub = [[] for _ in range(n_sp)]
    segs = anns if max_segments is None else anns[:max_segments]
    rng = np.random.default_rng(0)

    for rec in segs:
        ident = rec.get("identifier"); sp_name = split_of.get(ident)
        if sp_name is None:
            dropped["not_in_split"] += 1; continue
        label = int(rec.get("binary_label", 0)); action = rec.get("action_name", "")
        if is_pure_lower(action, label):
            dropped["pure_lower_body"] += 1; continue
        out = process_segment(rec, max_gap)
        if out is None:
            dropped["scale_fail"] += 1; continue
        speed, conf, sv, method = out
        subj = subject_of(ident); wins = 0
        cat0 = category(label, action)
        for start, x, ms, inv in iter_windows(speed, conf, sv, L, stride[sp_name], max_invalid):
            counts[sp_name] += 1; wins += 1
            meta[sp_name].append((ident, subj, action, label, cat0, start, ms, inv))
            if sp_name == "train" and rng.random() < 0.10:
                sb = x[:n_sp].T; vm = sv[start:start + L]
                for k in range(n_sp):
                    col = sb[:, k][vm[:, k]]
                    if col.size:
                        sub[k].append(col)
        if wins == 0:
            dropped["no_valid_window"] += 1
    if counts["train"] == 0:
        raise RuntimeError("No training windows produced.")

    med = np.zeros(n_sp); iqr = np.ones(n_sp); cap = np.full(n_sp, np.inf)
    for k in range(n_sp):
        if sub[k]:
            v = np.concatenate(sub[k])
            if v.size > 300000:
                v = rng.choice(v, 300000, replace=False)
            med[k] = np.median(v)
            iqr[k] = max(np.percentile(v, 75) - np.percentile(v, 25), 1e-6)
            cap[k] = np.percentile(v, 99.5)

    def alloc(name, n):
        mk = lambda nm, dt, sh: np.lib.format.open_memmap(
            os.path.join(out_dir, f"{nm}_{name}.npy"), mode="w+", dtype=dt, shape=sh)
        return (mk("X", np.float32, (n, C, L)), mk("y", np.int8, (n,)),
                mk("mask", np.uint8, (n, L, n_sp)), mk("F", np.float32, (n, n_spec)),
                mk("cat", np.int8, (n,)))
    handles = {nm: alloc(nm, counts[nm]) for nm in ("train", "test") if counts[nm]}
    idx = {nm: 0 for nm in handles}
    f_sum = np.zeros(n_spec); f_sqr = np.zeros(n_spec); f_cnt = 0

    for rec in segs:
        ident = rec.get("identifier"); sp_name = split_of.get(ident)
        if sp_name is None: continue
        label = int(rec.get("binary_label", 0)); action = rec.get("action_name", "")
        if is_pure_lower(action, label): continue
        out = process_segment(rec, max_gap)
        if out is None: continue
        speed, conf, sv, method = out
        if sp_name not in handles: continue
        X, y, M, F, CAT = handles[sp_name]
        cat0 = category(label, action)
        for start, x, ms, inv in iter_windows(speed, conf, sv, L, stride[sp_name], max_invalid):
            x = x.copy()
            sp_norm = x[:n_sp].T
            fvec = spectral(sp_norm)
            x[:n_sp] = ((np.minimum(x[:n_sp], cap[:, None]) - med[:, None]) / iqr[:, None])
            i = idx[sp_name]
            X[i] = x; y[i] = label; M[i] = sv[start:start + L].astype(np.uint8)
            F[i] = fvec; CAT[i] = cat0
            if sp_name == "train":
                f_sum += fvec; f_sqr += fvec ** 2; f_cnt += 1
            idx[sp_name] = i + 1

    f_mean = f_sum / max(f_cnt, 1)
    f_std = np.sqrt(np.maximum(f_sqr / max(f_cnt, 1) - f_mean ** 2, 1e-8))
    for nm in handles:
        F = handles[nm][3]
        F[:] = (F[:] - f_mean) / f_std
        for h in handles[nm]:
            h.flush()

    for nm in meta:
        if not meta[nm]: continue
        with open(os.path.join(out_dir, f"meta_{nm}.csv"), "w", newline="") as f:
            w = csv.writer(f); w.writerow(
                ["identifier", "subject", "action_name", "label", "cat",
                 "window_start", "mean_speed", "invalid_frac"])
            w.writerows(meta[nm])

    spec_names = [f"band{b}_{n}" for n in KEEP_NAMES for b in range(len(BANDS))]
    with open(os.path.join(out_dir, "channels.json"), "w") as f:
        json.dump({"layout": "(N,C,L) Conv1d-ready", "C": C, "L": L, "n_spec": n_spec,
                   "kept_keypoints": KEEP_NAMES,
                   "channels": [f"speed_{n}" for n in KEEP_NAMES] + [f"conf_{n}" for n in KEEP_NAMES],
                   "spectral_bands_hz": BANDS, "spectral_features": spec_names,
                   "cat_codes": {"0": "NoAction", "1": "gross-motor pos", "2": "fine-motor pos"},
                   "speed_scaling": "clip@p99.5 then (x-median)/IQR (robust)"}, f, indent=2)
    with open(os.path.join(out_dir, "scaler.json"), "w") as f:
        json.dump({"speed_median": med.tolist(), "speed_iqr": iqr.tolist(),
                   "speed_cap": cap.tolist(), "spec_mean": f_mean.tolist(),
                   "spec_std": f_std.tolist(), "keypoints": KEEP_NAMES}, f, indent=2)

    n_pos = sum(1 for r in meta["train"] if r[3] == 1)
    n_gross = sum(1 for r in meta["train"] if r[4] == 1)
    n_fine = sum(1 for r in meta["train"] if r[4] == 2)
    n_neg = counts["train"] - n_pos
    config = {"window": L, "stride_train": stride_train, "stride_test": stride_test,
              "max_gap": max_gap, "max_invalid": max_invalid,
              "n_train": counts["train"], "n_test": counts.get("test", 0),
              "train_neg": n_neg, "train_pos": n_pos,
              "train_gross_pos": n_gross, "train_fine_pos": n_fine,
              "pos_weight_v1": round(n_neg / max(n_pos, 1), 3),
              "pos_weight_v2": round(n_neg / max(n_gross, 1), 3),
              "dropped": dict(dropped)}
    with open(os.path.join(out_dir, "train_config.json"), "w") as f:
        json.dump(config, f, indent=2)

    lines = ["ASDMotion prep — quality report", "=" * 50,
             f"windows  train={counts['train']}  test={counts.get('test',0)}",
             f"train neg={n_neg} pos={n_pos} (gross={n_gross}, fine={n_fine})",
             f"pos_weight  V1={config['pos_weight_v1']}  V2={config['pos_weight_v2']}",
             f"dropped: {dict(dropped)}", ""]
    for lab in (0, 1):
        rows = [r for r in meta["train"] if r[3] == lab]
        if rows:
            lines.append(f"label {lab}: median invalid_frac="
                         f"{np.median([r[7] for r in rows]):.3f}")
    report = "\n".join(lines)
    with open(os.path.join(out_dir, "quality_report.txt"), "w") as f:
        f.write(report + "\n")
    return {"report": report, "config": config}

In [5]:
# ── Run prep + load arrays into memory ───────────────────────────────
res = prepare(PKL_PATH, OUT_DIR, WINDOW, STRIDE_TRAIN, STRIDE_TEST,
              MAX_GAP, MAX_INVALID, MAX_SEGMENTS)
print(res["report"])

def _load(split):
    return dict(
        X=np.load(f"{OUT_DIR}/X_{split}.npy", mmap_mode="r"),
        y=np.load(f"{OUT_DIR}/y_{split}.npy"),
        F=np.load(f"{OUT_DIR}/F_{split}.npy", mmap_mode="r"),
        cat=np.load(f"{OUT_DIR}/cat_{split}.npy"),
    )

train, test = _load("train"), _load("test")
cfg = res["config"]
print("\nX_train", train["X"].shape, " X_test", test["X"].shape,
      " F dim", train["F"].shape[1])
print("pos_weight  V1:", cfg["pos_weight_v1"], " V2:", cfg["pos_weight_v2"])

ASDMotion prep — quality report
windows  train=208130  test=11932
train neg=151429 pos=56701 (gross=24566, fine=32135)
pos_weight  V1=2.671  V2=6.164
dropped: {'no_valid_window': 3251, 'scale_fail': 3371, 'pure_lower_body': 395}

label 0: median invalid_frac=0.109
label 1: median invalid_frac=0.100

X_train (208130, 20, 90)  X_test (11932, 20, 90)  F dim 40
pos_weight  V1: 2.671  V2: 6.164
